# Proyecciones climáticas con datos grillados: climatología, anomalías y tendencia

**Geocomputación (1GEO20)** — PUCP, 2026-II — Semana 6


En este notebook procesamos datos grillados de proyecciones climáticas: cálculo de climatología, anomalías, promedio espacial y tendencia, para una región de estudio.

**Datos:** [NASA NEX-GDDP-CMIP6](https://developers.google.com/earth-engine/datasets/catalog/NASA_GDDP-CMIP6), variable `tasmax`, recortados a la región Lon(-85, -65) / Lat(-20, 5). 2 modelos (ACCESS-CM2, CMCC-ESM2), histórico (1950-2014) y 2 escenarios (SSP2-4.5, SSP5-8.5) hasta 2099, resolución mensual.

## 0. Descargar los datos 


In [ ]:
# Alternativa para descargar la carpeta compartida de Google Drive
# Descomenta y reemplaza el ID por el de la carpeta compartida

# !pip install gdown -q



In [ ]:
# # import gdown
# gdown.download_folder(id="ID_DE_LA_CARPETA_COMPARTIDA", output="datos_proyecciones", quiet=False)

# # o en Collab
# Descomenta y reemplaza el ID por el de la carpeta compartida
# !gdown --folder "ID_DE_LA_CARPETA_COMPARTIDA" -O datos_proyecciones



## 1. Setup y práctica con Xarray

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from scipy.stats import kendalltau, theilslopes

In [ ]:
# Ejemplo con un solo modelo y un solo escenario

carpeta_datos = "datos_proyecciones/"
modelo = "ACCESS-CM2"
escenario = "ssp585"

# 1. Cargar el histórico
ruta_historico = carpeta_datos + modelo + "/" + f"tasmax_{modelo}_1950-2014_historical.nc"
historico = xr.open_dataset(ruta_historico)

# 2. Cargar los dos tramos del escenario
carpeta_modelo = carpeta_datos + modelo + "/"
tramo1 = xr.open_dataset(carpeta_modelo + f"tasmax_{modelo}_2015-2049_{escenario}.nc")#["tasmax"]
tramo2 = xr.open_dataset(carpeta_modelo + f"tasmax_{modelo}_2050-2099_{escenario}.nc")#["tasmax"]



In [ ]:
# 3. Concatenar todo en una sola variable continua 1950-2099
completo_ejemplo = xr.concat([historico, tramo1, tramo2], dim="time")
print(completo_ejemplo.sizes, completo_ejemplo.time.min().values, "a", completo_ejemplo.time.max().values)

In [ ]:
# 4. Climatología: agrupa los pasos de tiempo por mes del calendario
# (todos los eneros juntos, todos los febreros juntos, ...) y promedia cada grupo
climatologia_ejemplo = completo_ejemplo.groupby("time.month").mean("time")
print(climatologia_ejemplo.sizes)  # queda (month: 12, lat, lon): un campo promedio por cada mes del año

In [ ]:
climatologia_ejemplo

## 2. Cargar todos los datos y armar series continuas

Cada modelo tiene su propia subcarpeta, con el histórico y dos tramos por escenario:

```
datos_proyecciones/
├── ACCESS-CM2/
│   ├── tasmax_ACCESS-CM2_1950-2014_historical.nc
│   ├── tasmax_ACCESS-CM2_2015-2049_ssp245.nc
│   ├── tasmax_ACCESS-CM2_2050-2099_ssp245.nc
│   ├── tasmax_ACCESS-CM2_2015-2049_ssp585.nc
│   └── tasmax_ACCESS-CM2_2050-2099_ssp585.nc
└── CMCC-ESM2/
    └── ... (misma estructura)
```

El histórico es el mismo para ambos escenarios de un modelo, así que lo cargamos **una sola vez por modelo** y lo combinamos con cada escenario.

In [ ]:
carpeta_datos = "datos_proyecciones/"

modelos = ["ACCESS-CM2", "CMCC-ESM2"]
escenarios = ["ssp245", "ssp585"]  # tal cual aparecen en los nombres de archivo


In [ ]:


def cargar_historico(modelo, carpeta):
    ruta = carpeta + modelo + f"/tasmax_{modelo}_1950-2014_historical.nc"
    return xr.open_dataset(ruta)["tasmax"]


def cargar_escenario(modelo, escenario, carpeta):
    carpeta_modelo = carpeta + modelo + "/"
    tramo1 = xr.open_dataset(carpeta_modelo + f"tasmax_{modelo}_2015-2049_{escenario}.nc")["tasmax"]
    tramo2 = xr.open_dataset(carpeta_modelo + f"tasmax_{modelo}_2050-2099_{escenario}.nc")["tasmax"]
    return xr.concat([tramo1, tramo2], dim="time")

In [ ]:
da_por_modelo = []
for modelo in modelos:
    historico = cargar_historico(modelo, carpeta_datos)
    da_por_escenario = []
    for escenario in escenarios:
        futuro = cargar_escenario(modelo, escenario, carpeta_datos)
        da_por_escenario.append(xr.concat([historico, futuro], dim="time"))
    serie_modelo = xr.concat(da_por_escenario, dim="escenario")
    serie_modelo = serie_modelo.assign_coords(escenario=escenarios)
    da_por_modelo.append(serie_modelo)

datos_completo = xr.concat(da_por_modelo, dim="modelo")
datos_completo = datos_completo.assign_coords(modelo=modelos)

In [ ]:
datos_completo

## 3. Climatología y anomalía

Calculamos la climatología mensual (promedio por mes dentro de un periodo base) y la anomalía respecto a esa climatología, a nivel de grilla completa (todavía sin promediar espacialmente).

El periodo base queda como parámetro modificable, más abajo lo van a cambiar ustedes en el Ejercicio 1.

In [ ]:
periodo_base = (1985, 2014)  # años inicio y fin, ambos incluidos


def calcular_climatologia_anomalia(da, periodo_base):
    inicio, fin = periodo_base
    base = da.sel(time=slice(f"{inicio}-01-01", f"{fin}-12-31"))
    climatologia = base.groupby("time.month").mean("time")
    anomalia = da.groupby("time.month") - climatologia
    return climatologia, anomalia


climatologia, anomalia = calcular_climatologia_anomalia(datos_completo, periodo_base)

In [ ]:
anomalia

## 4. Inspección rápida (sin cartopy)

Revisar visualmente que el dominio esté bien y que la anomalía tenga sentido espacial, con  `.plot()` de xarray (sin mapa base, solo para inspección rápida).

In [ ]:
# Dominio: un mes cualquiera del histórico, para confirmar la extensión y forma de la grilla
climatologia.sel(modelo="ACCESS-CM2",escenario="ssp245").isel(month=0).plot()
plt.show()

In [ ]:
# Anomalía espacial: promedio de los últimos 10 años del siglo, SSP5-8.5
anomalia.sel(modelo="ACCESS-CM2",escenario="ssp245", time=slice("2090-01-01", "2099-12-31")).mean("time").plot(
    cmap="RdBu_r", center=0, levels=np.arange(-10,10), extend='both',
)
plt.show()

### Ejercicio 1

Vuelve a calcular climatología y anomalía para ` "ACCESS-CM2" - "ssps585")`, esta vez con un periodo base distinto (por ejemplo, todo el histórico 1950-2014 en vez de 1985-2014).

Compara la anomalía de un mismo mes (por ejemplo junio de 2090) entre ambos periodos base. ¿Cuánto cambia? 

In [ ]:
# tu código aquí


## 5. Promedio espacial (ponderado por latitud)

Para pasar de la grilla a una sola serie de tiempo por modelo y escenario, promediamos sobre el espacio ponderando la latitud ([.wighted()](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.weighted.html)). La región es opcional, si no se especifica se usa todo el dominio descargado, pero se puede acotar a una subregión dentro de él.

In [ ]:
def promedio_espacial(dataarray, lon_min=None, lon_max=None, lat_min=None, lat_max=None):
    sub = dataarray
    if lon_min is not None or lon_max is not None:
        sub = sub.sel(x=slice(lon_min, lon_max))
    if lat_min is not None or lat_max is not None:
        sub = sub.sel(y=slice(lat_max, lat_min))

    # ponderamos por cos(lat): una celda cerca del ecuador cubre más área real
    # que una celda cerca de los 20°S, aunque ambas midan lo mismo en grados
    pesos = np.cos(np.deg2rad(sub["y"]))
    return sub.weighted(pesos).mean(dim=("y", "x"))




In [ ]:
# todo el dominio descargado

series_regionales = promedio_espacial(anomalia)

# Prueba para una subregión opcional (descomenta para probar)
# sub = promedio_espacial(anomalia.sel(modelo="ACCESS-CM2", escenario="ssps585"), lon_min=-80, lon_max=-70, lat_min=-15, lat_max=0)

In [ ]:
series_regionales.sel(modelo="ACCESS-CM2",escenario="ssp245")

## 6. De mensual a anual

In [ ]:
series_anuales = series_regionales.groupby("time.year").mean("time")

In [ ]:
series_anuales.sel(modelo="ACCESS-CM2",escenario="ssp245").plot()

## 7. Test de tendencia (prueba)

 `kendalltau` para significancia, `theilslopes` para la magnitud.

In [ ]:
def interpretar_tendencia(tau, p_valor, alpha=0.05):
    if p_valor < alpha and tau > 0:
        return "creciente"
    elif p_valor < alpha and tau < 0:
        return "decreciente"
    return "sin tendencia significativa"


serie_prueba = series_anuales.sel(modelo="ACCESS-CM2", escenario="ssp585")
anios = serie_prueba["year"].values
valores = serie_prueba.values

tau, p_valor = kendalltau(anios, valores)
pendiente, intercepto, ic_bajo, ic_alto = theilslopes(valores, anios, alpha=0.95)

print(f"ACCESS-CM2, ssp585: tau={tau:.3f}, p-valor={p_valor:.2e} -> {interpretar_tendencia(tau, p_valor)}")
print(f"Pendiente de Sen: {pendiente*10:.3f} °C/década (IC 95%: [{ic_bajo*10:.3f}, {ic_alto*10:.3f}])")

### Ejercicio 2

Con el ejemplo de arriba como guía y el notebook `proyecciones_climaticas.ipynb`, aplica `kendalltau` y `theilslopes` a las **4 combinaciones** de `series_anuales` (2 modelos × 2 escenarios) en una interación, y junta los resultados en una tabla (`pd.DataFrame`).

¿Qué tanto difiere SSP2-4.5 de SSP5-8.5 dentro de un mismo modelo?

In [ ]:
# tu código aquí

#### Ensemble mean

In [ ]:
ensemble_mean = series_anuales.mean("modelo")   # o anomalia.mean("modelo") para el dataarray de anomalías (time,x,y)

## 8. Gráfico final

Con lo que ya procesaron: anomalía anual de `tasmax`, por modelo y escenario.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

estilo_color = {"ACCESS-CM2": "tab:blue", "CMCC-ESM2": "tab:orange"}
estilo_linea = {"ssp245": "-", "ssp585": "--"}

for modelo in series_anuales.modelo.values:
    for escenario in series_anuales.escenario.values:
        serie = series_anuales.sel(modelo=modelo, escenario=escenario)
        ax.plot(serie["year"], serie.values,
            color=estilo_color[modelo], linestyle=estilo_linea[escenario],
            label=f"{modelo} {escenario}")

ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Año")
ax.set_ylabel("Anomalía tasmax (°C)")
ax.legend(fontsize=8)
plt.show()

### Ejercicio 3

Grafica la media del ensamble (los 2 modelos) para cada escenario, junto con el rango entre ambos modelos (mínimo-máximo, no una desviación estándar, con solo 2 modelos no tiene sentido estadístico real).

Pista: `series_anuales.mean("modelo")`, `.min("modelo")` y `.max("modelo")`, luego `fill_between` entre el mínimo y el máximo.

In [ ]:
# tu código aquí

### Ejercicio 4 

Grafica un mapa con cartopy comparando el cambio espacial de `tasmax` entre escenarios: por ejemplo, la anomalía media 2070-2099 de SSP2-4.5 junto a la de SSP5-8.5 (dos subplots), para uno de los dos modelos.

Pistas: usa `plt.subplots(1, 2, subplot_kw={"projection": ccrs.PlateCarree()})`, y el patrón de `ax.contourf(...)` + `ax.coastlines()`.

In [ ]:
# tu código aquí